# K-Means Time Series Analysis on Hindi Romance Films

Here I want to analyze the variation of color throughout the length of films. Basically as before splitting a film into 5 segments, applying k_means to that segments, getting 15 ish colors, but unlike before i won't group all the colors together, rather ill analyze each segment separately.

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML 
import matplotlib.colors as mcolors
import math
from collections import Counter
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

### Load the data and preprocess

In [2]:
# Path to the directory containing the CSV files
directory_path = '/Users/rsudhir/Documents/GitHub/Data-Science-Project---Colors-Of-Romance/Hindi-Analysis/Hindi-Movie-CSVs'

# List to hold each DataFrame
dfs = []

# Loop through the files in the directory and load each CSV
for i in range(1, 26):
    file_path = os.path.join(directory_path, f'{i}.csv')
    df = pd.read_csv(file_path)
    # Drop columns with NaN values
    df = df.drop(columns=['color_10_r', 'color_10_g', 'color_10_b'])
    
    # Add a column to indicate which movie the data is from
    df['movie_id'] = i
    
    dfs.append(df)

# Combine all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# Display the combined DataFrame to ensure it looks correct
display(combined_df.head())

,frame_path,color_1_r,color_1_g,color_1_b,color_2_r,color_2_g,color_2_b,color_3_r,color_3_g,color_3_b,...,color_7_r,color_7_g,color_7_b,color_8_r,color_8_g,color_8_b,color_9_r,color_9_g,color_9_b,movie_id
0,output_0000001.png,117,55,10,232,219,96,231,90,12,...,136,91,33,180,162,100,180,173,96,1
1,output_0000002.png,4,4,4,8,4,4,8,4,4,...,8,4,4,8,4,4,8,4,4,1
2,output_0000003.png,199,163,104,17,17,8,170,130,77,...,27,69,69,108,140,132,100,148,134,1
3,output_0000004.png,168,147,89,106,90,47,9,9,6,...,187,5,47,89,122,140,44,61,73,1
4,output_0000005.png,37,49,32,186,199,163,135,139,109,...,88,106,100,130,162,156,192,26,50,1


In [3]:
# Define the number of segments
num_segments = 5

# Loop through each movie and create a segment ID
combined_df['segment_id'] = combined_df.groupby('movie_id').cumcount() // (combined_df.groupby('movie_id')['frame_path'].transform('count') // num_segments)

### K_means on each segment

For each segment of each movie i am getting x colors

In [4]:
# Apply K-Means Clustering to Each Segment Separately
num_clusters_per_segment = 15  # Number of clusters per segment

# Create a list to hold cluster centers for each segment
segment_clusters = []

# Group by movie and segment
grouped = combined_df.groupby(['movie_id', 'segment_id'])

for (movie_id, segment_id), group in grouped:
    # Extract RGB values for the dominant colors in this segment
    colors = []
    for i in range(1, 10):
        colors.append(group[[f'color_{i}_r', f'color_{i}_g', f'color_{i}_b']].dropna().values)
    
    # Combine colors into a single array
    colors_array = np.vstack(colors)

    # Ensure sufficient data points for clustering
    if len(colors_array) >= num_clusters_per_segment:
        # Perform K-Means clustering
        kmeans = KMeans(n_clusters=num_clusters_per_segment, n_init='auto', random_state=42)
        kmeans.fit(colors_array)
        
        # Store the cluster centers for this segment
        segment_clusters.append((movie_id, segment_id, kmeans.cluster_centers_))
    else:
        print(f"Skipping segment {segment_id} of movie {movie_id} due to insufficient data points.")


Skipping segment 5 of movie 1 due to insufficient data points.
Skipping segment 5 of movie 3 due to insufficient data points.
Skipping segment 5 of movie 5 due to insufficient data points.
Skipping segment 5 of movie 9 due to insufficient data points.
Skipping segment 5 of movie 12 due to insufficient data points.


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1152: ConvergenceWarning: Number of distinct clusters (11) found smaller than n_clusters (15). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


### Visualization of all segments and their colors

Have commented out the code as its quite a long output, feel free to uncomment if you'd like to see all the colors.

In [5]:
# # Visualization and Analysis
# for movie_id, segment_id, centers in segment_clusters:
#     # Convert RGB values to HEX
#     hex_colors = [mcolors.to_hex([r/255, g/255, b/255]) for r, g, b in centers]

#     # Sort colors by hue for better visualization
#     hsv_colors = [mcolors.rgb_to_hsv([r/255, g/255, b/255]) for r, g, b in centers]
#     sorted_indices = sorted(range(len(hsv_colors)), key=lambda i: (hsv_colors[i][0], hsv_colors[i][1], hsv_colors[i][2]))
#     sorted_hex_colors = [hex_colors[i] for i in sorted_indices]

#     # Create an HTML table for the colors
#     num_columns = 5  # Adjust as needed
#     num_rows = math.ceil(len(sorted_hex_colors) / num_columns)
#     html_table = '<table style="border-collapse: collapse;">'
#     for i in range(num_rows):
#         html_table += '<tr>'
#         for j in range(num_columns):
#             index = i * num_columns + j
#             if index < len(sorted_hex_colors):
#                 hex_color = sorted_hex_colors[index]
#                 html_table += f'<td style="background-color:{hex_color}; width:50px; height:25px; border: 1px solid #ccc;"></td>'
#                 html_table += f'<td style="padding: 5px;">{hex_color}</td>'
#         html_table += '</tr>'
#     html_table += '</table>'
    
#     # Display the HTML table with the colors for this segment
#     display(HTML(f"<h3>Movie {movie_id} - Segment {segment_id}</h3>"))
#     display(HTML(html_table))


### K_means Across Grouped Segments

In [6]:
# Group by segment_id across all movies
combined_segments = {}

for segment_id in range(num_segments):
    segment_colors = []
    for movie_id, _, centers in segment_clusters:
        if segment_id == _:
            segment_colors.append(centers)
    
    # Combine all colors from this segment across movies into a single array
    combined_segments[segment_id] = np.vstack(segment_colors)

In [7]:
# Define the number of clusters for each combined segment
num_clusters_per_combined_segment = 30  # Adjust this as needed

# Store the final cluster centers for each combined segment
final_segment_clusters = {}

for segment_id, colors in combined_segments.items():
    if len(colors) >= num_clusters_per_combined_segment:
        # Perform K-Means clustering on the combined colors for this segment
        kmeans = KMeans(n_clusters=num_clusters_per_combined_segment, n_init='auto', random_state=42)
        kmeans.fit(colors)
        
        # Store the cluster centers for this segment
        final_segment_clusters[segment_id] = kmeans.cluster_centers_
    else:
        print(f"Skipping combined segment {segment_id} due to insufficient data points.")

### Visualizing Colors Across Segments

In [8]:
# Visualize the colors for each combined segment
for segment_id, centers in final_segment_clusters.items():
    # Convert RGB values to HEX
    hex_colors = [mcolors.to_hex([r/255, g/255, b/255]) for r, g, b in centers]

    # Sort colors by hue for better visualization
    hsv_colors = [mcolors.rgb_to_hsv([r/255, g/255, b/255]) for r, g, b in centers]
    sorted_indices = sorted(range(len(hsv_colors)), key=lambda i: (hsv_colors[i][0], hsv_colors[i][1], hsv_colors[i][2]))
    sorted_hex_colors = [hex_colors[i] for i in sorted_indices]

    # Create an HTML table for the colors
    num_columns = 5  # Adjust as needed
    num_rows = math.ceil(len(sorted_hex_colors) / num_columns)
    html_table = '<table style="border-collapse: collapse;">'
    for i in range(num_rows):
        html_table += '<tr>'
        for j in range(num_columns):
            index = i * num_columns + j
            if index < len(sorted_hex_colors):
                hex_color = sorted_hex_colors[index]
                html_table += f'<td style="background-color:{hex_color}; width:50px; height:25px; border: 1px solid #ccc;"></td>'
                html_table += f'<td style="padding: 5px;">{hex_color}</td>'
        html_table += '</tr>'
    html_table += '</table>'
    
    # Display the HTML table with the colors for this segment
    display(HTML(f"<h3>Combined Segment {segment_id}</h3>"))
    display(HTML(html_table))

,#a5332e,,#a75742,,#53392b,,#6a5849,,#c5b19d
,#d2c5b6,,#b7b5b3,,#84735a,,#a69478,,#d4ae6a
,#8c8472,,#525048,,#f2c52f,,#302f28,,#171715
,#a1a197,,#1c9d59,,#d3d9d7,,#5f6c6c,,#328ca5
,#74abbc,,#2f4553,,#3d576e,,#818e9a,,#9db2c5
,#5f7d9d,,#2538b1,,#8f3d99,,#bc313c,,#c55b5d


,#ad3d35,,#9a5840,,#a44c1e,,#5a3d2e,,#937461
,#bb9476,,#c7b09b,,#735d47,,#362f27,,#1a1815
,#8f8a76,,#c6c5ba,,#b3ad45,,#cdcf72,,#a1a38f
,#525851,,#d3d8d7,,#687070,,#2e4344,,#369aa9
,#628197,,#448abf,,#96a6b6,,#85919e,,#3c4f6b
,#aab8cf,,#4b54ae,,#ce4b9c,,#b84f5b,,#bf2d3a


,#b03c36,,#a96045,,#553b2b,,#bd997f,,#8c6f58
,#9c8976,,#c5b7a7,,#68563b,,#dbbb4b,,#bca137
,#191814,,#cccdc2,,#2e3027,,#5e6258,,#789c4e
,#9fa29d,,#777e76,,#485150,,#65b6cf,,#3991ad
,#35464d,,#606b70,,#5b8296,,#426c8d,,#d1d7dd
,#355068,,#80909f,,#9eb3ca,,#487cbe,,#c1235f


,#b2423e,,#872d21,,#5d4335,,#97654a,,#593925
,#a98e77,,#796858,,#c59b75,,#362e23,,#645643
,#c7b8a3,,#181715,,#c6ab3e,,#a5a99a,,#93a85e
,#578339,,#8d918b,,#6e766d,,#374241,,#525f61
,#398ba3,,#d0d6d8,,#62b2d5,,#38495b,,#2d5686
,#a4b3c7,,#6282b4,,#8894a9,,#64718d,,#967575


,#d71a1a,,#b03d3a,,#85504c,,#854025,,#d9d7d6
,#a55e43,,#997b6a,,#462c1a,,#9e9188,,#d0b49e
,#564130,,#71553e,,#c8976b,,#716758,,#171513
,#f3f2ef,,#aaa395,,#2e2d27,,#dbc826,,#808777
,#4f5658,,#397ba5,,#6aa4cd,,#9dadb9,,#2e4558
,#5e7081,,#8290a1,,#bebebe,,#6636ab,,#c3437b
